# Hands-on Activity 8.1: Aggregating Data with Pandas <br>
8. 1.1 Intended Learning Outcomes
After this activity, the student should be able to:
Demonstrate querying and merging of dataframes
Perform advanced calculations on dataframes
Aggregate dataframes with pandas and numpy
Work with time series data <br>
8. 1.2 Resources
Computing Environment using Python 3.x
Attached Datasets (under Instructional Materials)
8. 1.3 Procedures
The procedures can be found in the canvas module. Check the following under topics:
* 1 Weather Data Collection
* 2 Querying and Merging
* 3 Dataframe Operations
* 4 Aggregations
* 5 Time Series
* Data Analysis

1. With the earthquakes.csv file, select all the earthquakes in Japan with a magType of mb and a magnitude of 4.9 or greater.

In [1]:
import pandas as pd
import numpy as np

In [3]:
import pandas as pd

def load_datasets():

    eq = pd.read_csv("earthquakes.csv", parse_dates=["time"])
    stocks = pd.read_csv("faang.csv", parse_dates=["date"])

    return eq, stocks


earthquakes, faang = load_datasets()


print(earthquakes.head())
print(faang.head())

    mag magType           time                  place  tsunami parsed_place
0  1.35      ml  1539475168010  9km NE of Aguanga, CA        0   California
1  1.29      ml  1539475129610  9km NE of Aguanga, CA        0   California
2  3.42      ml  1539475062610  8km NE of Aguanga, CA        0   California
3  0.44      ml  1539474978070  9km NE of Aguanga, CA        0   California
4  2.16      md  1539474716050  10km NW of Avenal, CA        0   California
  ticker       date    open    high       low   close    volume
0     FB 2018-01-02  177.68  181.58  177.5500  181.42  18151903
1     FB 2018-01-03  181.88  184.78  181.3300  184.67  16886563
2     FB 2018-01-04  184.90  186.21  184.0996  184.33  13880896
3     FB 2018-01-05  185.59  186.90  184.9300  186.85  13574535
4     FB 2018-01-08  187.20  188.90  186.3300  188.28  17994726


/tmp/ipython-input-1258/3170045307.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  eq = pd.read_csv("earthquakes.csv", parse_dates=["time"])



2. Create bins for each full number of magnitude (for example, the first bin is 0-1, the second is 1-2, and so on) with a magType of ml and count how many are in each bin.

In [4]:
japan_mb = earthquakes.loc[
    earthquakes["place"].str.contains("Japan", na=False)
    & earthquakes["magType"].eq("mb")
    & earthquakes["mag"].ge(4.9)
]

print(japan_mb)

      mag magType           time                         place  tsunami  \
1563  4.9      mb  1538977532250  293km ESE of Iwo Jima, Japan        0   
2576  5.4      mb  1538697528010    37km E of Tomakomai, Japan        0   
3072  4.9      mb  1538579732490     15km ENE of Hasaki, Japan        0   
3632  4.9      mb  1538450871260    53km ESE of Hitachi, Japan        0   

     parsed_place  
1563        Japan  
2576        Japan  
3072        Japan  
3632        Japan  



3. Using the faang.csv file, group by the ticker and resample to monthly frequency. Make the following aggregations:
* Mean of the opening price
* Maximum of the high price
* Minimum of the low price
* Mean of the closing price
* Sum of the volume traded


In [5]:
faang.set_index('date', inplace=True)

monthly_agg = (faang.groupby('ticker').resample('ME').agg({'open': 'mean', 'high': 'max', 'low': 'min', 'close': 'mean', 'volume': 'sum' }))


4. Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum
magnitude that was observed for each combination. Put the magType along the columns.

In [6]:
crosstab_max = pd.crosstab(earthquakes['tsunami'], earthquakes['magType'], values=earthquakes['mag'], aggfunc='max')


5. Calculate the rolling 60-day aggregations of OHLC data by ticker for the FAANG data. Use the same aggregations as exercise no. 3.


In [9]:

earthquakes, faang = load_datasets()


rolling_60 = (
    faang
        .groupby("ticker")
        .rolling(window=60)
        .agg({
            "open": "mean",
            "high": "max",
            "low": "min",
            "close": "mean",
            "volume": "sum"
        })
)


print(
    crosstab_max,
    "\n",
    monthly_agg.head(),
    "\n",
    rolling_60.head()
)

magType   mb  mb_lg    md   mh   ml  ms_20    mw  mwb  mwr  mww
tsunami                                                        
0        5.6    3.5  4.11  1.1  4.2    NaN  3.83  5.8  4.8  6.0
1        6.1    NaN   NaN  NaN  5.1    5.7  4.41  NaN  NaN  7.5 
                          open      high       low       close     volume
ticker date                                                             
AAPL   2018-01-31  170.714690  176.6782  161.5708  170.699271  659679440
       2018-02-28  164.562753  177.9059  147.9865  164.921884  927894473
       2018-03-31  172.421381  180.7477  162.4660  171.878919  713727447
       2018-04-30  167.332895  176.2526  158.2207  167.286924  666360147
       2018-05-31  182.635582  187.9311  162.7911  183.207418  620976206 
             open  high  low  close  volume
ticker                                    
AAPL   251   NaN   NaN  NaN    NaN     NaN
       252   NaN   NaN  NaN    NaN     NaN
       253   NaN   NaN  NaN    NaN     NaN
       254   N

/tmp/ipython-input-1258/3170045307.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  eq = pd.read_csv("earthquakes.csv", parse_dates=["time"])



6. Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.

In [10]:
pivot_table = pd.pivot_table(faang.reset_index(),values=['open', 'high', 'low', 'close', 'volume'], index='ticker', aggfunc='mean')
print(pivot_table)

              close         high          low         open        volume
ticker                                                                  
AAPL     186.986218   188.906858   185.135729   187.038674  3.402145e+07
AMZN    1641.726175  1662.839801  1619.840398  1644.072669  5.649563e+06
FB       171.510936   173.615298   169.303110   171.454424  2.768798e+07
GOOG    1113.225139  1125.777649  1101.001594  1113.554104  1.742645e+06
NFLX     319.290299   325.224583   313.187273   319.620533  1.147030e+07


7. Calculate the Z-scores for each numeric column of Netflix's data (ticker is NFLX) using apply().

In [11]:

nflx = faang.loc[faang["ticker"].eq("NFLX")]

cols = ["open", "high", "low", "close", "volume"]
z_scores = (nflx[cols] - nflx[cols].mean()) / nflx[cols].std()

print(z_scores.head())

         open      high       low     close    volume
753 -2.500753 -2.516023 -2.410226 -2.416644 -0.088760
754 -2.380291 -2.423180 -2.285793 -2.335286 -0.507606
755 -2.296272 -2.406077 -2.234616 -2.323429 -0.959287
756 -2.275014 -2.345607 -2.202087 -2.234303 -0.782331
757 -2.218934 -2.295113 -2.143759 -2.192192 -1.038531


8. Add event descriptions:
Create a dataframe with the following three columns: ticker, date, and event. The columns should have the following values:
ticker: 'FB'
date: ['2018-07-25', '2018-03-19', '2018-03-20']
event: ['Disappointing user growth announced after close.', 'Cambridge Analytica story', 'FTC investigation']
Set the index to ['date', 'ticker']
Merge this data with the FAANG data using an outer join

In [16]:

events.set_index(['date', 'ticker'], inplace=True)
faang_reset = faang.reset_index()
faang_reset.set_index(['date', 'ticker'], inplace=True)

merged = faang_reset.merge(events, how='outer', left_index=True, right_index=True)
merged.head()

index       open       high        low      close  \
date       ticker                                                      
2018-01-02 AAPL      251   166.9271   169.0264   166.0442   168.9872   
           AMZN      502  1172.0000  1190.0000  1170.5100  1189.0100   
           FB          0   177.6800   181.5800   177.5500   181.4200   
           GOOG     1004  1048.3400  1066.9400  1045.2300  1065.0000   
           NFLX      753   196.1000   201.6500   195.4200   201.0700   

                     volume event  
date       ticker                  
2018-01-02 AAPL    25555934   NaN  
           AMZN     2694494   NaN  
           FB      18151903   NaN  
           GOOG     1237564   NaN  
           NFLX    10966889   NaN


9. Use the transform() method on the FAANG data to represent all the values in terms of the first date in the data. To do so, divide all the values for each ticker by the values
for the first date in the data for that ticker. This is referred to as an index, and the data for the first date is the base (https://ec.europa.eu/eurostat/statistics-explained/
index.php/ Beginners:Statisticalconcept-Indexandbaseyear). When data is in this format, we can easily see growth over time. Hint: transform() can take a function name.

In [17]:
indexed = (faang.groupby('ticker')[['open', 'high', 'low', 'close', 'volume']] .transform(lambda x: x / x.iloc[0]))

indexed.head()

,open,high,low,close,volume
0,1.000000,1.000000,1.000000,1.000000,1.000000
1,1.023638,1.017623,1.021290,1.017914,0.930292
2,1.040635,1.025498,1.036889,1.016040,0.764707
3,1.044518,1.029298,1.041566,1.029931,0.747830
4,1.053579,1.040313,1.049451,1.037813,0.991341
